# ETL pipeline assignment

1. Upload the provided files to your Databricks account on a DBFS storage.
2. Create an ETL pipeline that does the following using Spark Notebooks:

- Read the parquet files from DBFS storage and create bronze table for each individual file using append only operation.
o	Create silver layer table that uses SCD1 type. Each bronze table will be mapped to an individual silver table.
- Create gold layer tables to create analytical queries based on the below requirements. A total of 3 gold tables would be needed.
  1.	Get the most sold products to identify the top-selling items.
  2.	Find which suppliers provide ingredients to the most franchises.
  3.	Get total sales per month.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS BRONZE;
CREATE SCHEMA IF NOT EXISTS SILVER;
CREATE SCHEMA IF NOT EXISTS GOLD;

### Bronze Layer

In [0]:
%sql
CREATE TABLE IF NOT EXISTS default.file_load_log (
  file_name STRING,
  load_time TIMESTAMP
)
USING DELTA;

In [0]:
def load_newfiles_bronze(filename, table_name, schema="bronze", file_format="parquet"):
    file_path = f"dbfs:/FileStore/tables/{filename}.parquet"
    schema_location = "/schema_tracking"

    result = spark.sql(f"""
        SELECT 1 
        FROM default.file_load_log
        WHERE file_name = '{file_path}'
        LIMIT 1
    """)

    already_loaded = result.count() > 0

    if already_loaded:
        return f"File {file_path} already loaded."
    else:
        autoloader_df = (spark.read.format(file_format)\
            .option("cloudFiles.format", file_format)\
            .option("cloudFiles.schemaLocation", schema_location)\
            .load(file_path))
        
        autoloader_df.write.format("delta")\
            .mode("append")\
            .saveAsTable(f"{schema}.{table_name}")

        spark.sql(f"""
        INSERT INTO default.file_load_log (file_name, load_time)
        VALUES ('{file_path}', current_timestamp())
        """)

        return autoloader_df

In [0]:
load_newfiles_bronze("media_gold_reviews_chunked", "media_gold_reviews_chunked")
load_newfiles_bronze("media_customer_reviews", "media_customer_reviews")
load_newfiles_bronze("sales_customers", "sales_customers")
load_newfiles_bronze("sales_franchises", "sales_franchises")
load_newfiles_bronze("sales_suppliers", "sales_suppliers")
load_newfiles_bronze("sales_transactions", "sales_transactions")

Out[164]: 'File dbfs:/FileStore/tables/sales_transactions.parquet already loaded.'

### Silver layer _SCD1_

In [0]:
from delta.tables import DeltaTable

def load_data_silver(table_name, unique_col, source_schema="bronze", target_schema="silver"):
  if not spark.catalog.tableExists(f"{target_schema}.{table_name}"):
    spark.sql(f"""
        CREATE TABLE {target_schema}.{table_name}
        AS (select * from {source_schema}.{table_name} where 1=0)
    """)

  bronze_df = spark.read.table(f"{source_schema}.{table_name}") 
  silver_table = DeltaTable.forName(spark, f"{target_schema}.{table_name}")
  condition = f"{target_schema}.{unique_col} = {source_schema}.{unique_col}"

  merge_result = silver_table.alias(target_schema).merge(
      bronze_df.alias(source_schema),
      condition
  ).whenMatchedUpdateAll()\
  .whenNotMatchedInsertAll()\
  .execute()

In [0]:
load_data_silver("media_gold_reviews_chunked", "franchiseID")
load_data_silver("media_customer_reviews", "franchiseID")
load_data_silver("sales_customers", "customerID")
load_data_silver("sales_franchises", "franchiseID")
load_data_silver("sales_suppliers", "supplierID")
load_data_silver("sales_transactions", "transactionID")

## Gold Layer

### 1.	Get the most sold products to identify the top-selling items.

In [0]:
from pyspark.sql.functions import sum, col, current_date, date_sub, count

# Load sales_transactions data 
df = spark.read.table("silver.sales_transactions")

# Get the most sold products to identify the top-selling items
top_selling_items = df.groupBy("product") \
    .agg(sum("quantity").alias("total_product_sold")) \
    .orderBy(col("total_product_sold").desc())

# Save result as a table 
top_selling_items.write.format("delta").mode("overwrite").saveAsTable("gold.top_selling_products")


In [0]:
%sql
select * from gold.top_selling_products;

product,total_product_sold
Golden Gate Ginger,3865
Outback Oatmeal,3733
Austin Almond Biscotti,3716
Tokyo Tidbits,3662
Pearly Pies,3595
Orchard Oasis,3586


### 2.	Find which suppliers provide ingredients to the most franchises.

In [0]:
# table: sales_franchises
from pyspark.sql.functions import sum, col, current_date, date_sub, count, collect_set

# Load sales_franchises data 
df = spark.read.table("silver.sales_franchises")

# Get the which suppliers provide ingredients to the most franchises
top_selling_items = df.groupBy("franchiseID" ,"supplierID") \
    .agg(count("*").alias("supplier_franchise_count")) \
    .orderBy(col("supplier_franchise_count").desc())

# Save result as a table 
top_selling_items.write.format("delta").mode("overwrite").saveAsTable("gold.supplier_franchise_details")


In [0]:
%sql
select * from gold.supplier_franchise_details;

franchiseID,supplierID,supplier_franchise_count
3000029,4000029,1
3000012,4000012,1
3000023,4000023,1
3000030,4000030,1
3000043,4000043,1
3000004,4000004,1
3000017,4000017,1
3000021,4000021,1
3000038,4000038,1
3000013,4000013,1


### 3.	Get total sales per month.

In [0]:
# table: sales_franchises
from pyspark.sql.functions import sum, col, year, month

# Load sales_transactions data
df = spark.read.table("silver.sales_transactions")

# Get total sales per month
top_selling_items = df.groupBy(year("dateTime").alias("year"), month("dateTime").alias("month")) \
    .agg(sum("totalPrice").alias("sales_per_month")) \
    .orderBy(col("sales_per_month").desc())

# Save result as a table 
top_selling_items.write.format("delta").mode("overwrite").saveAsTable("gold.sales_per_month")


In [0]:
%sql
select * from gold.sales_per_month;

year,month,sales_per_month
2024,5,66471
